In [ ]:
# IMPORTS
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# DATA LOADING
def load_data(path: str) -> pd.DataFrame:
  return pd.read_csv(path)

In [ ]:
# PREPROCESSING (simple, non-ML transforms)
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()

  # Drop constant / ID-like columns if present
  drop_cols = ["EmployeeCount","StandardHours","Over18","EmployeeNumber"]
  df = df.drop(columns=[c for c in drop_cols if c in df.columns], error="ignore")

  # Encode target
  df["Attrition"] = df["Attrition"].map({"Yes":1, "No":0})

  return df

In [ ]:
# TRAIN-TEST SPLIT
def split_data(df:pd.DataFrame, target: str):
  X = df.drop(target, axis=1)
  y = df[target]
  return train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [ ]:
# MODEL BUILDING
def build_model(categorical_cols, numeric_cols):
  preprocessor = ColumnTransformer(
      transformers=[
          ("num", StandardScaler(), numeric_cols),
          ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
      ]
  )

  model = Pipeline(
      steps=[
          ("preprocessor", preprocessor),
          ("classifier", RandomForestClassifier(random_state=42))
      ]
  )

  return model

In [ ]:
# TRAINING
from sklearn.model_selection import GridSearchCV
def train_model(model, X_train, y_train):

  param_grid = {
      "classifier__n_estimators":[100,200,300],
      "classifier__max_depth":[None, 5, 10, 15],
      "classifier__min_samples_split": [2,5,10],
      "classifier__min_samples_leaf" : [1,2,4]
  }

  grid = GridSearchCV(
      estimator=model,
      param_grid = param_grid,
      cv = 3,
      n_jobs=1,
      scoring = "accuracy"
  )

  grid.fit(X_train, y_train)

  print("\nBest Params:", grid.best_params_)
  print("Best Score from CV:", grid.best_score_)




  return grid.best_estimator_

In [ ]:
# EVALUATION
def evaluate(model, X_test, y_test):
  y_pred = model.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  print(f"Accuracy : {acc:.3f}")
  print("\nClassification Report:\n")
  print(classification_report(y_test, y_pred))

In [ ]:
# PREPROCESSING (simple, non-ML transforms)
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()

  # Drop constant / ID-like columns if present
  drop_cols = ["EmployeeCount","StandardHours","Over18","EmployeeNumber"]
  df = df.drop(columns=[c for c in drop_cols if c in df.columns])

  # Encode target
  df["Attrition"] = df["Attrition"].map({"Yes":1, "No":0})

  return df

# MAIN PIPELINE
def main():

  df = load_data("/content/hr_data.csv")

  df = preprocess(df)

  X_train, X_test, y_train, y_test = split_data(df, target="Attrition")

  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
  numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

  model = build_model(categorical_cols, numerical_cols)



  best_model = train_model(model, X_train, y_train)



  evaluate(best_model, X_test, y_test)



  print(categorical_cols)
  print(numerical_cols)

if __name__ == "__main__":
  main()


Best Params: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best Score from CV: 0.8681972789115647
Accuracy : 0.844

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.98      0.91       247
           1       0.55      0.13      0.21        47

    accuracy                           0.84       294
   macro avg       0.70      0.55      0.56       294
weighted avg       0.81      0.84      0.80       294

['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLi

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
def train_model_random_Search(model, X_train, y_train):

  param_dist = {
      "classifier__n_estimators":[100,200,300,500],
      "classifier__max_depth":[None, 5, 10, 20],
      "classifier__min_samples_split":[2,5,10],
      "classifier__min_samples_leaf":[1,2,4],
      "classifier__max_features":["auto","sqrt","log2"]
  }

  rand_search = RandomizedSearchCV(
      estimator=model,
      param_distributions=param_dist,
      n_iter=20,
      cv=3,
      scoring="accuracy",
      n_jobs=-1,
      random_state=42
  )

  rand_search.fit(X_train, y_train)

  print("\nBest params:", rand_search.best_params_)
  print("Best CV Score:", rand_search.best_score_)
  return rand_search.best_estimator_

In [ ]:
# PREPROCESSING (simple, non-ML transforms)
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()

  # Drop constant / ID-like columns if present
  drop_cols = ["EmployeeCount","StandardHours","Over18","EmployeeNumber"]
  df = df.drop(columns=[c for c in drop_cols if c in df.columns])

  # Encode target
  df["Attrition"] = df["Attrition"].map({"Yes":1, "No":0})

  return df

# MAIN PIPELINE
def main():

  df = load_data("/content/hr_data.csv")

  df = preprocess(df)

  X_train, X_test, y_train, y_test = split_data(df, target="Attrition")

  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
  numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

  model = build_model(categorical_cols, numerical_cols)

  randomized_model = train_model_random_Search(model, X_train, y_train)







  evaluate(randomized_model, X_test, y_test)


if __name__ == "__main__":
  main()

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
9 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last


Best params: {'classifier__n_estimators': 500, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'sqrt', 'classifier__max_depth': None}
Best CV Score: 0.8647959183673469
Accuracy : 0.844

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.98      0.91       247
           1       0.55      0.13      0.21        47

    accuracy                           0.84       294
   macro avg       0.70      0.55      0.56       294
weighted avg       0.81      0.84      0.80       294



In [ ]:

from xgboost import XGBClassifier
def xgb_model(categorical_cols, numerical_cols):

  preprocesor = ColumnTransformer(
      transformers = [
          ("num", StandardScaler(), numerical_cols),
          ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
      ]
  )

  xgb = XGBClassifier(
      objective="binary:logistic",
      eval_metric = "logloss",
      random_state = 42
  )

  model = Pipeline(
      steps = [
          ("preprocessor",preprocesor),
          ("xgbclassifier",xgb)
      ]
  )

  return model

In [ ]:
# PREPROCESSING (simple, non-ML transforms)
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()

  # Drop constant / ID-like columns if present
  drop_cols = ["EmployeeCount","StandardHours","Over18","EmployeeNumber"]
  df = df.drop(columns=[c for c in drop_cols if c in df.columns])

  # Encode target
  df["Attrition"] = df["Attrition"].map({"Yes":1, "No":0})

  return df

# MAIN PIPELINE
def main():

  df = load_data("/content/hr_data.csv")

  df = preprocess(df)

  X_train, X_test, y_train, y_test = split_data(df, target="Attrition")

  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
  numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

  model = xgb_model(categorical_cols, numerical_cols)

  # Train the model before evaluation
  model.fit(X_train, y_train)

  evaluate(model, X_test, y_test)


if __name__ == "__main__":
  main()

Accuracy : 0.857

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.97      0.92       247
           1       0.63      0.26      0.36        47

    accuracy                           0.86       294
   macro avg       0.75      0.61      0.64       294
weighted avg       0.83      0.86      0.83       294

